# Exploratory data analysis: Telco Customer Churn

Goal: understand who churns before building models. Questions I want to answer:

1. How imbalanced is the target?
2. Which columns need cleaning?
3. How do contract type, tenure, charges and payment method relate to churn?
4. Do the engineered features (tenure buckets, charges per month of tenure, number of services) separate churners from non-churners?

The dataset must be in `data/` first (see `data/README.md`).

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from churn import config
from churn.data import clean, load_raw
from churn.features import add_features

sns.set_theme(style="whitegrid")
raw = load_raw(config.RAW_DATA_PATH)
raw.head()

## 1. Shape, types and missing values

`TotalCharges` is read as text. The blank values belong to customers with `tenure == 0`, who have not been billed yet.

In [ ]:
raw.info()

In [ ]:
blank_total = raw["TotalCharges"].str.strip() == ""
print("Blank TotalCharges:", blank_total.sum())
raw.loc[blank_total, ["tenure", "MonthlyCharges", "TotalCharges"]]

## 2. Target balance

Churners are a minority, so accuracy alone is misleading: a model that predicts "no churn" for everyone already gets a high accuracy. This is why F1 on the churn class is the main metric.

In [ ]:
df = add_features(clean(raw))
df["Churn"].value_counts(normalize=True).rename("share")

## 3. Churn rate by categorical drivers

In [ ]:
def churn_rate(column: str) -> pd.Series:
    return df.groupby(column)["Churn"].mean().sort_values(ascending=False)

drivers = [
    "Contract", "PaymentMethod", "InternetService",
    "TechSupport", "PaperlessBilling", "SeniorCitizen",
]
for column in drivers:
    print(churn_rate(column).round(3), end="\n\n")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, column in zip(axes, ["Contract", "PaymentMethod", "tenure_bucket"]):
    churn_rate(column).plot.bar(ax=ax, title=f"Churn rate by {column}")
    ax.set_ylabel("churn rate")
    ax.tick_params(axis="x", rotation=30)
plt.tight_layout()

## 4. Numeric features

Compare the distributions of tenure and charges for churners and non-churners.

In [ ]:
numeric = ["tenure", "MonthlyCharges", "TotalCharges", "charges_per_tenure_month", "num_services"]
fig, axes = plt.subplots(1, len(numeric), figsize=(20, 4))
for ax, column in zip(axes, numeric):
    sns.kdeplot(data=df, x=column, hue="Churn", common_norm=False, fill=True, ax=ax)
plt.tight_layout()

In [ ]:
df[numeric + ["contract_months", "Churn"]].corr()["Churn"].sort_values()

## Notes for modelling

Things to check in the outputs above and carry into `src/churn/features.py`:

- churn is concentrated in month-to-month contracts and in the first year of tenure;
- electronic check payers and fiber optic customers churn more;
- `TotalCharges` is strongly correlated with tenure, so the ratio `charges_per_tenure_month` is more informative than the raw total;
- the target is imbalanced, so models use class weights and the decision threshold is tuned for F1.